# PCB-MC figures

Visual / interactive cells extracted from `Results_YOLO_RTDETR.ipynb`,
`5.1 D-FINE_Results.ipynb` and `Confusion_Matrix_Missing_Components.ipynb`
that don't belong in `scripts/reproduce_paper_tables.py`: per-image
GT/prediction overlays and confusion-matrix heatmap rendering.

The cross-fold numeric aggregation (mean/std per metric across folds,
confusion-matrix counts, per-class precision/recall/FNR) now lives in
`scripts/reproduce_paper_tables.py` and is imported here rather than
re-derived.

Paths below are the original Colab (`/content/drive/MyDrive/PCB_MC/...`)
paths from the source notebooks — update `DATA_ROOT` / `RESULTS_BASE_DIR`
for your environment before running.

## D-FINE: per-image GT vs. prediction overlays

From `5.1 D-FINE_Results.ipynb`.

In [ ]:
!pip install -q opencv-python matplotlib pandas

In [ ]:
def read_coco_gt(gt_json_path):
    """Load a COCO-format ground-truth JSON.
    Returns:
      - images: dict image_id -> {file_name,width,height}
      - categories: dict category_id -> category_name
      - annotations_by_image: dict image_id -> list of {category_id, bbox[x,y,w,h]}
    """
    import json, os
    if not os.path.exists(gt_json_path):
        raise FileNotFoundError(f"GT COCO JSON not found: {gt_json_path}")

    with open(gt_json_path, "r") as f:
        data = json.load(f)

    images = {img["id"]: img for img in data.get("images", [])}
    categories = {cat["id"]: cat["name"] for cat in data.get("categories", [])}

    annotations_by_image = {}
    for ann in data.get("annotations", []):
        img_id = ann["image_id"]
        annotations_by_image.setdefault(img_id, []).append({
            "category_id": ann["category_id"],
            "bbox": ann["bbox"],  # [x,y,w,h]
        })
    return images, categories, annotations_by_image


def read_coco_predictions(pred_json_path):
    """Load predictions in COCO 'detections' format.

    Accepts either:
      - a list of dicts: [{image_id, category_id, bbox[x,y,w,h], score}, ...]
      - or a dict with a key holding that list (common: 'annotations'/'predictions'/'results')

    Returns dict image_id -> list of {category_id,bbox,score}
    """
    import json, os
    if not os.path.exists(pred_json_path):
        raise FileNotFoundError(f"Predictions JSON not found: {pred_json_path}")

    with open(pred_json_path, "r") as f:
        data = json.load(f)

    # normalize to list
    if isinstance(data, dict):
        for k in ["annotations", "predictions", "results", "detections"]:
            if k in data and isinstance(data[k], list):
                data = data[k]
                break

    if not isinstance(data, list):
        raise ValueError("Predictions JSON must be a list of detections or a dict containing such a list.")

    preds_by_image = {}
    for det in data:
        img_id = det.get("image_id")
        if img_id is None:
            continue
        preds_by_image.setdefault(img_id, []).append({
            "category_id": det.get("category_id"),
            "bbox": det.get("bbox"),   # [x,y,w,h]
            "score": det.get("score", None),
        })
    return preds_by_image

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# NOTE: read_coco_gt and read_coco_predictions are defined in the preceding cell.

# =========================
# CONFIG (EDIT THESE PATHS)
# =========================
IMAGES_DIR      = "/content/drive/MyDrive/PCB_MC/Data"             # folder containing PCB images
GT_COCO_JSON    = "/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/valid/COCO_valid.json" # COCO GT json (images/categories/annotations)
RESULTS_BASE_DIR = "/content/drive/MyDrive/PCB_MC/Results/D-Fine"   # Base directory for D-Fine results

# Subsets to iterate through
SUBSETS = ["components_only", "full_dataset", "missing_only", "non_missing"]

# Filtering / thresholds
CONF_THRESHOLD = 0.25         # ignore predictions below this
IOU_THRESHOLD  = 0.50         # TP matching threshold
SHOW_TEXT_BELOW_CONF = 0.80   # show text for non-missing TPs only if score < this
MAX_IMAGES = 12               # how many images to visualize (set None for all)

# Drawing controls
FONT_SCALE_LABELS = 0.45
TEXT_THICKNESS = 1
BOX_THICKNESS = 2
BOX_THICKNESS_CRITICAL = 3
FILL_ALPHA = 0.30             # 0 = no fill, 0.25-0.45 recommended for dense boards

# =========================
# LABEL ABBREVIATIONS
# =========================
CLASS_ABBREV = {
    "Button": "BTN",
    "Capacitor": "C",
    "Electrolytic Capacitor": "EC",
    "Resistor": "R",
    "Inductor": "L",
    "Ferrite Bead": "FB",
    "Diode": "D",
    "Zener Diode": "ZD",
    "Led": "LED",
    "IC": "IC",
    "Connector": "CONN",
    "Switch": "SW",
    "Test Point": "TP",
    "Pins": "PIN",
    "Pads": "PAD",
    "Clock": "CLK",
    "Display": "DISP",
    "Fuse": "FUSE",
    "Heatsink": "HS",
    "Potentiometer": "POT",
    "Jumper": "JMP",
    "EM": "EM",
}

def _abbr(class_name: str) -> str:
    return CLASS_ABBREV.get(class_name, class_name)

def _is_missing(class_name: str) -> bool:
    return str(class_name).lower().startswith("missing")

def _format_label(class_name: str, conf, prefix: str = "") -> str:
    base = _abbr(class_name)
    if prefix:
        base = f"{prefix}{base}"
    if conf is None:
        return base
    return f"{base} {conf:.2f}"

# =========================
# SEMANTIC HIGH-CONTRAST COLORS (RGB)
# =========================
COLOR_IDENTIFIED = (255, 0, 255)  # Magenta (complementary to green)
COLOR_CONNECTOR  = (255, 255, 0)  # Yellow
COLOR_MISSING    = (255, 0, 0)    # Red
COLOR_FP         = (255, 165, 0)  # Orange
COLOR_FN         = (255, 0, 0)    # Red
COLOR_NEUTRAL    = (255, 255, 255)

def color_for_class(class_name: str):
    if class_name == "Connector":
        return COLOR_CONNECTOR
    if _is_missing(class_name):
        return COLOR_MISSING
    if class_name in ["Pins", "Pads"]:
        return COLOR_NEUTRAL
    return COLOR_IDENTIFIED

# =========================
# DRAWING UTILITIES
# =========================
def _draw_text_with_outline(img, text, org, font_scale):
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                (0, 0, 0), thickness=TEXT_THICKNESS + 2, lineType=cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                (255, 255, 255), thickness=TEXT_THICKNESS, lineType=cv2.LINE_AA)

def draw_bbox(image, bbox_xyxy, label="", color=(255,0,255), thickness=2, fill_alpha=0.0):
    x1, y1, x2, y2 = [int(v) for v in bbox_xyxy]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(image.shape[1]-1, x2), min(image.shape[0]-1, y2)

    if fill_alpha and fill_alpha > 0:
        overlay = image.copy()
        cv2.rectangle(overlay, (x1,y1), (x2,y2), color, thickness=-1)
        image[:] = cv2.addWeighted(overlay, fill_alpha, image, 1-fill_alpha, 0)

    cv2.rectangle(image, (x1,y1), (x2,y2), color, thickness=thickness)

    if label:
        y_text = y1 - 6 if y1 - 6 > 10 else y1 + 14
        _draw_text_with_outline(image, label, (x1, y_text), FONT_SCALE_LABELS)

def xywh_to_xyxy(b):
    x, y, w, h = b
    return [x, y, x+w, y+h]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    iw = max(0.0, inter_x2 - inter_x1)
    ih = max(0.0, inter_y2 - inter_y1)
    inter = iw * ih
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union

def greedy_match(preds, gts, iou_thr=0.5):
    """Greedy one-to-one matching by highest IoU.
    preds: list of dict with 'bbox_xyxy'
    gts:   list of dict with 'bbox_xyxy'
    Returns:
      matches: list of (pred_idx, gt_idx, iou)
      unmatched_pred_idxs, unmatched_gt_idxs
    """
    if not preds or not gts:
        return [], list(range(len(preds))), list(range(len(gts)))

    iou_mat = np.zeros((len(preds), len(gts)), dtype=np.float32)
    for i,p in enumerate(preds):
        for j,g in enumerate(gts):
            iou_mat[i,j] = iou_xyxy(p["bbox_xyxy"], g["bbox_xyxy"])

    matches = []
    used_p, used_g = set(), set()
    while True:
        best = (-1, -1, 0.0)
        for i in range(len(preds)):
            if i in used_p:
                continue
            for j in range(len(gts)):
                if j in used_g:
                    continue
                v = float(iou_mat[i,j])
                if v > best[2]:
                    best = (i, j, v)
        if best[0] == -1 or best[2] < iou_thr:
            break
        i, j, v = best
        used_p.add(i); used_g.add(j)
        matches.append((i, j, v))

    unmatched_p = [i for i in range(len(preds)) if i not in used_p]
    unmatched_g = [j for j in range(len(gts)) if j not in used_g]
    return matches, unmatched_p, unmatched_g

# =========================
# LOAD GLOBAL GT DATA (once)
# =========================
images, categories, gt_by_image = read_coco_gt(GT_COCO_JSON)
cat_name_to_id = {v:k for k,v in categories.items()}

print(f"Loaded GT images: {len(images)} | categories: {len(categories)}")

# =========================
# VISUALIZE FOR EACH SUBSET
# =========================
for subset_name in SUBSETS:
    print(f"\nProcessing subset: {subset_name}")

    PRED_JSON_PATH = os.path.join(RESULTS_BASE_DIR, subset_name, "predictions.json")
    OUTPUT_DIR     = f"./dfine_vis_out_{subset_name}"

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    try:
        pred_by_image = read_coco_predictions(PRED_JSON_PATH)
        print(f"Loaded predictions for {len(pred_by_image)} images from {PRED_JSON_PATH}")
    except FileNotFoundError:
        print(f"Predictions JSON not found for {subset_name} at {PRED_JSON_PATH}. Skipping this subset.")
        continue
    except ValueError as e:
        print(f"Error loading predictions for {subset_name}: {e}. Skipping this subset.")
        continue

    all_image_ids = sorted(images.keys())
    if MAX_IMAGES is not None:
        all_image_ids = all_image_ids[:MAX_IMAGES]

    for img_id in all_image_ids:
        img_info = images[img_id]
        img_path = os.path.join(IMAGES_DIR, img_info["file_name"])
        if not os.path.exists(img_path):
            alt_path = os.path.join(IMAGES_DIR, os.path.basename(img_info["file_name"]))
            if os.path.exists(alt_path):
                img_path = alt_path
            else:
                print(f"Missing image file: {img_info['file_name']}")
                continue

        bgr = cv2.imread(img_path)
        if bgr is None:
            print(f"Could not read: {img_path}")
            continue
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        gt_list = gt_by_image.get(img_id, [])
        pred_list = pred_by_image.get(img_id, [])

        preds = []
        for p in pred_list:
            score = p.get("score", None)
            if score is not None and score < CONF_THRESHOLD:
                continue
            bbox = p.get("bbox", None)
            if bbox is None:
                continue
            preds.append({
                "category_id": p.get("category_id"),
                "score": score,
                "bbox_xyxy": xywh_to_xyxy(bbox)
            })

        gts = []
        for g in gt_list:
            bbox = g.get("bbox", None)
            if bbox is None:
                continue
            gts.append({
                "category_id": g.get("category_id"),
                "bbox_xyxy": xywh_to_xyxy(bbox)
            })

        matched_pred = set()
        matched_gt = set()
        canvas = img.copy()

        for cat_id, cat_name in categories.items():
            preds_c = [(idx,p) for idx,p in enumerate(preds) if p["category_id"] == cat_id]
            gts_c   = [(idx,g) for idx,g in enumerate(gts)   if g["category_id"] == cat_id]

            if not preds_c and not gts_c:
                continue

            preds_pack = [p for _,p in preds_c]
            gts_pack = [g for _,g in gts_c]

            matches, un_p, un_g = greedy_match(preds_pack, gts_pack, iou_thr=IOU_THRESHOLD)

            for pi, gi, iou_v in matches:
                global_pi = preds_c[pi][0]
                global_gi = gts_c[gi][0]
                matched_pred.add(global_pi)
                matched_gt.add(global_gi)

                p = preds[global_pi]
                score = p["score"]
                cname = cat_name

                col = color_for_class(cname)
                thick = BOX_THICKNESS_CRITICAL if _is_missing(cname) else BOX_THICKNESS

                show_text = _is_missing(cname) or (score is not None and score < SHOW_TEXT_BELOW_CONF)
                label = _format_label(cname, score) if show_text else ""
                draw_bbox(canvas, p["bbox_xyxy"], label=label, color=col, thickness=thick, fill_alpha=FILL_ALPHA if _is_missing(cname) else 0.0)

            for gi in un_g:
                global_gi = gts_c[gi][0]
                if global_gi in matched_gt:
                    continue
                g = gts[global_gi]
                cname = cat_name
                label = _format_label(cname, None, prefix="FN ")
                draw_bbox(canvas, g["bbox_xyxy"], label=label, color=COLOR_FN, thickness=BOX_THICKNESS_CRITICAL, fill_alpha=FILL_ALPHA)

        for i,p in enumerate(preds):
            if i in matched_pred:
                continue
            cname = categories.get(p["category_id"], str(p["category_id"]))
            score = p.get("score", None)
            label = _format_label(cname, score, prefix="FP ")
            draw_bbox(canvas, p["bbox_xyxy"], label=label, color=COLOR_FP, thickness=BOX_THICKNESS, fill_alpha=0.0)

        plt.figure(figsize=(10, 8))
        plt.imshow(canvas)
        plt.axis("off")
        plt.title(f"D-FINE Visualization | Subset: {subset_name} | image_id={img_id} | {img_info['file_name']}")
        plt.show()

        out_name = Path(img_info["file_name"]).name
        out_path = os.path.join(OUTPUT_DIR, out_name)
        cv2.imwrite(out_path, cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))

    print(f"Saved visualizations for {subset_name} to: {os.path.abspath(OUTPUT_DIR)}")

print("\nAll subsets processed.")

## YOLOv11: per-image TP/FP/FN overlays

From `Results_YOLO_RTDETR.ipynb` ("Visualization YOLOV11" section).

In [ ]:
!pip install ultralytics --upgrade -q

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import yaml

# -----------------------------------------------------------------------------
# Visualization tuned for green PCBs (YOLO outputs in normalized xywh)
# Goals:
# - High-contrast colors against green solder mask
# - Less clutter: show text only for Missing/*, FNs, and low-confidence preds
# - Readable text: black outline + white text
# - Optional alpha-blended boxes for dense regions
# -----------------------------------------------------------------------------

FONT_SCALE_LABELS = 0.45
TEXT_THICKNESS = 1
BOX_THICKNESS = 3
BOX_THICKNESS_CRITICAL = 3
SHOW_TEXT_BELOW_CONF = 0.80
FILL_ALPHA = 0.30

CLASS_ABBREV = {
    "Button": "BTN", "Capacitor": "C", "Electrolytic Capacitor": "EC", "Resistor": "R",
    "Inductor": "L", "Ferrite Bead": "FB", "Diode": "D", "Zener Diode": "ZD", "Led": "LED",
    "IC": "IC", "Connector": "CONN", "Switch": "SW", "Test Point": "TP", "Pins": "PIN",
    "Pads": "PAD", "Clock": "CLK", "Display": "DISP", "Fuse": "FUSE", "Heatsink": "HS",
    "Potentiometer": "POT", "Jumper": "JMP", "EM": "EM",
}

def _is_missing(class_name: str) -> bool:
    return class_name.lower().startswith("missing")

def _abbr(class_name: str) -> str:
    return CLASS_ABBREV.get(class_name, class_name)

def _format_label(class_name: str, conf, prefix: str = "") -> str:
    base = _abbr(class_name)
    if prefix:
        base = f"{prefix}{base}"
    if conf is None:
        return base
    return f"{base} {conf:.2f}"

COLOR_IDENTIFIED = (255, 0, 255)
COLOR_CONNECTOR  = (255, 255, 0)
COLOR_MISSING    = (255, 0, 0)
COLOR_FP         = (255, 165, 0)
COLOR_FN         = (255, 0, 0)
COLOR_NEUTRAL    = (255, 255, 255)

class_color_map = {
    "Button": COLOR_IDENTIFIED, "Capacitor": COLOR_IDENTIFIED, "Clock": COLOR_IDENTIFIED,
    "Diode": COLOR_IDENTIFIED, "Display": COLOR_IDENTIFIED, "EM": COLOR_IDENTIFIED,
    "Electrolytic Capacitor": COLOR_IDENTIFIED, "Ferrite Bead": COLOR_IDENTIFIED,
    "Fuse": COLOR_IDENTIFIED, "Heatsink": COLOR_IDENTIFIED, "IC": COLOR_IDENTIFIED,
    "Inductor": COLOR_IDENTIFIED, "Jumper": COLOR_IDENTIFIED, "Led": COLOR_IDENTIFIED,
    "Potentiometer": COLOR_IDENTIFIED, "Resistor": COLOR_IDENTIFIED, "Switch": COLOR_IDENTIFIED,
    "Test Point": COLOR_IDENTIFIED, "Transistor": COLOR_IDENTIFIED, "Zener Diode": COLOR_IDENTIFIED,
    "Connector": COLOR_CONNECTOR,
    "Pads": COLOR_NEUTRAL, "Pins": COLOR_NEUTRAL,
    "Missing Component": COLOR_MISSING, "Missing IC": COLOR_MISSING, "Missing Resistor": COLOR_MISSING,
    "Missing Capacitor": COLOR_MISSING, "Missing Diode": COLOR_MISSING, "Missing Ferrite Bead": COLOR_MISSING,
    "Missing Inductor": COLOR_MISSING, "Missing Led": COLOR_MISSING,
}

def _draw_text_with_outline(img, text, org, font_scale):
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                (0, 0, 0), thickness=TEXT_THICKNESS + 2, lineType=cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                (255, 255, 255), thickness=TEXT_THICKNESS, lineType=cv2.LINE_AA)

def draw_bbox(image, bbox, label="", color=(255, 0, 255),
              line_thickness=3, font_scale=0.5, fill_alpha=0.0):
    """Draw a bbox (normalized xywh) with optional alpha-filled background and readable text."""
    h, w, _ = image.shape
    x_center, y_center, bbox_w, bbox_h = bbox

    x_min = int((x_center - bbox_w / 2) * w)
    y_min = int((y_center - bbox_h / 2) * h)
    x_max = int((x_center + bbox_w / 2) * w)
    y_max = int((y_center + bbox_h / 2) * h)

    x_min = max(0, x_min); y_min = max(0, y_min)
    x_max = min(w - 1, x_max); y_max = min(h - 1, y_max)

    if fill_alpha and fill_alpha > 0:
        overlay = image.copy()
        cv2.rectangle(overlay, (x_min, y_min), (x_max, y_max), color, thickness=-1)
        cv2.addWeighted(overlay, fill_alpha, image, 1 - fill_alpha, 0, dst=image)

    cv2.rectangle(image, (x_min, y_min), (x_max, y_max), color, line_thickness)

    if label:
        (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, TEXT_THICKNESS)
        tx = x_min
        ty = y_min - 6
        if ty - th < 0:
            ty = y_min + th + 6
        _draw_text_with_outline(image, label, (tx, ty), font_scale)

    return image

def draw_true_positives(image, true_positives, class_names_list, class_color_map):
    annotated_image = image.copy()
    for tp in true_positives:
        class_name = class_names_list[tp["class_id"]]
        color = class_color_map.get(class_name, COLOR_IDENTIFIED)
        conf = float(tp["confidence"])
        show_text = _is_missing(class_name) or conf < SHOW_TEXT_BELOW_CONF
        label = _format_label(class_name, conf) if show_text else ""
        thickness = BOX_THICKNESS_CRITICAL if _is_missing(class_name) else BOX_THICKNESS
        fill = FILL_ALPHA if _is_missing(class_name) else 0.0
        annotated_image = draw_bbox(
            annotated_image, tp["pred_box"], label=label, color=color,
            line_thickness=thickness, font_scale=FONT_SCALE_LABELS, fill_alpha=fill
        )
    return annotated_image

def draw_false_positives(image, false_positives, class_names_list, class_color_map):
    annotated_image = image.copy()
    for fp in false_positives:
        class_name = class_names_list[fp["class_id"]]
        conf = float(fp["confidence"])
        label = _format_label(class_name, conf, prefix="FP ")
        annotated_image = draw_bbox(
            annotated_image, fp["pred_box"], label=label, color=COLOR_FP,
            line_thickness=BOX_THICKNESS, font_scale=FONT_SCALE_LABELS, fill_alpha=0.0
        )
    return annotated_image

def draw_false_negatives(image, false_negatives, class_names_list, class_color_map):
    annotated_image = image.copy()
    for fn in false_negatives:
        class_name = class_names_list[fn["class_id"]]
        label = _format_label(class_name, None, prefix="FN ")
        thickness = BOX_THICKNESS_CRITICAL
        annotated_image = draw_bbox(
            annotated_image, fn["gt_box"], label=label, color=COLOR_FN,
            line_thickness=thickness, font_scale=FONT_SCALE_LABELS, fill_alpha=FILL_ALPHA
        )
    return annotated_image

def get_gt_annotations(label_file_path, img_width, img_height):
    """Reads YOLO-format GT annotations. gt_box is normalized [x_center, y_center, w, h]."""
    gt_annotations = []
    if not os.path.exists(label_file_path):
        return gt_annotations
    with open(label_file_path, "r") as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            class_id = int(parts[0])
            gt_annotations.append({"class_id": class_id, "gt_box": parts[1:]})
    return gt_annotations

def calculate_iou(boxA, boxB):
    """IoU of two normalized [x_center, y_center, w, h] boxes."""
    xA1, yA1 = boxA[0] - boxA[2] / 2, boxA[1] - boxA[3] / 2
    xA2, yA2 = boxA[0] + boxA[2] / 2, boxA[1] + boxA[3] / 2
    xB1, yB1 = boxB[0] - boxB[2] / 2, boxB[1] - boxB[3] / 2
    xB2, yB2 = boxB[0] + boxB[2] / 2, boxB[1] + boxB[3] / 2

    x_intersect_min = max(xA1, xB1)
    y_intersect_min = max(yA1, yB1)
    x_intersect_max = min(xA2, xB2)
    y_intersect_max = min(yA2, yB2)

    inter_area = max(0, x_intersect_max - x_intersect_min) * max(0, y_intersect_max - y_intersect_min)
    boxA_area = boxA[2] * boxA[3]
    boxB_area = boxB[2] * boxB[3]
    denom = boxA_area + boxB_area - inter_area
    return inter_area / denom if denom > 0 else 0

def visualize_detections(img_path, model, class_names_list, labels_dir, iou_threshold=0.5):
    """Runs inference and splits detections into TP/FP/FN vs. YOLO-format GT."""
    true_positives, false_positives, false_negatives = [], [], []

    results = model(img_path, verbose=False)
    predictions = results[0]

    img_temp = cv2.imread(img_path)
    if img_temp is None:
        print(f"Error: Could not load image {img_path} for GT processing.")
        return [], [], []
    img_height, img_width, _ = img_temp.shape

    base_img_name = os.path.basename(img_path).rsplit(".", 1)[0]
    label_file_path = os.path.join(labels_dir, f"{base_img_name}.txt")
    gt_annotations = get_gt_annotations(label_file_path, img_width, img_height)

    pred_boxes_xywhn = predictions.boxes.xywhn.cpu().numpy()
    pred_confs = predictions.boxes.conf.cpu().numpy()
    pred_class_ids = predictions.boxes.cls.cpu().numpy().astype(int)

    matched_gt_indices = [False] * len(gt_annotations)

    for i in range(len(pred_class_ids)):
        pred_box = pred_boxes_xywhn[i]
        pred_class_id = pred_class_ids[i]
        pred_conf = pred_confs[i]

        best_iou = 0
        best_gt_idx = -1
        for j, gt_ann in enumerate(gt_annotations):
            if matched_gt_indices[j]:
                continue
            iou = calculate_iou(pred_box, gt_ann["gt_box"])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j

        if best_iou >= iou_threshold and best_gt_idx != -1 and gt_annotations[best_gt_idx]["class_id"] == pred_class_id:
            true_positives.append({
                "class_id": pred_class_id, "confidence": pred_conf, "pred_box": pred_box,
                "gt_box": gt_annotations[best_gt_idx]["gt_box"],
            })
            matched_gt_indices[best_gt_idx] = True
        else:
            false_positives.append({"class_id": pred_class_id, "confidence": pred_conf, "pred_box": pred_box})

    for j, gt_ann in enumerate(gt_annotations):
        if not matched_gt_indices[j]:
            false_negatives.append({"class_id": gt_ann["class_id"], "gt_box": gt_ann["gt_box"]})

    return true_positives, false_positives, false_negatives

def load_class_names_from_yaml(yaml_path):
    if not os.path.exists(yaml_path):
        print(f"Warning: data.yaml not found at {yaml_path}")
        return []
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    return data.get("names", [])

print("Visualization functions defined.")

In [ ]:
ROOT_DIR = '/content/drive/MyDrive/PCB_MC/Data/'
VISUALIZATION_RESULTS_DIR = '/content/yolov11_visualizations'
os.makedirs(VISUALIZATION_RESULTS_DIR, exist_ok=True)

subsets = ['components_only', 'full_dataset', 'missing_only', 'non_missing']

results_dirs = {
    'components_only': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only',
    'full_dataset': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset',
    'missing_only': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only',
    'non_missing': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing',
}

# Select representative image file paths: prefer images common to all subsets
num_images_to_select = 12
all_subset_image_names = {}
for subset in subsets:
    images_path = os.path.join(ROOT_DIR, subset, 'kfold_data', 'fold_1', 'valid', 'images')
    if os.path.exists(images_path):
        all_subset_image_names[subset] = set([f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    else:
        all_subset_image_names[subset] = set()

common_image_names = set.intersection(*all_subset_image_names.values()) if all_subset_image_names else set()
selected_common_image_names = list(common_image_names)[:num_images_to_select]

selected_images_per_subset = {}
for subset in subsets:
    images_path = os.path.join(ROOT_DIR, subset, 'kfold_data', 'fold_1', 'valid', 'images')
    subset_images_to_process = [
        os.path.join(images_path, img_name)
        for img_name in selected_common_image_names
        if img_name in all_subset_image_names[subset]
    ]
    if len(subset_images_to_process) < num_images_to_select:
        available_images = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png')) and f not in selected_common_image_names]
        subset_images_to_process.extend([os.path.join(images_path, img_name) for img_name in available_images[:(num_images_to_select - len(subset_images_to_process))]])

    selected_images_per_subset[subset] = subset_images_to_process[:num_images_to_select]
    print(f"Selected {len(selected_images_per_subset[subset])} images for {subset}")

for subset, image_paths in selected_images_per_subset.items():
    if not image_paths:
        print(f"No images to process for subset: {subset}")
        continue

    print(f"\nProcessing subset: {subset}")

    weights_path = os.path.join(results_dirs[subset], 'fold_1', 'weights', 'best.pt')
    if not os.path.exists(weights_path):
        print(f"Weights not found for {subset} fold_1 at {weights_path}. Skipping.")
        continue

    print(f"Loading model for {subset} from {weights_path}")
    model = YOLO(weights_path)

    labels_dir = os.path.join(ROOT_DIR, subset, 'kfold_data', 'fold_1', 'valid', 'labels')
    if not os.path.exists(labels_dir):
        print(f"Labels directory not found for {subset} fold_1 at {labels_dir}. Skipping.")
        continue

    subset_output_dir = os.path.join(VISUALIZATION_RESULTS_DIR, subset)
    os.makedirs(subset_output_dir, exist_ok=True)
    print(f"Saving visualizations for {subset} to {subset_output_dir}")

    data_yaml_path = os.path.join(ROOT_DIR, subset, 'kfold_data', 'fold_0', 'data.yaml')
    current_class_names = load_class_names_from_yaml(data_yaml_path)
    if not current_class_names:
        print(f"Could not load class names from {data_yaml_path}. Skipping visualization for {subset}.")
        continue

    for img_path in image_paths:
        print(f"  Visualizing: {os.path.basename(img_path)}")
        original_image = cv2.imread(img_path)
        if original_image is None:
            print(f"Error: Could not load image {img_path}")
            continue
        original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

        true_positives, false_positives, false_negatives = visualize_detections(
            img_path, model, current_class_names, labels_dir
        )

        img_tp = draw_true_positives(original_image_rgb.copy(), true_positives, current_class_names, class_color_map)
        img_fp = draw_false_positives(original_image_rgb.copy(), false_positives, current_class_names, class_color_map)
        img_fn = draw_false_negatives(original_image_rgb.copy(), false_negatives, current_class_names, class_color_map)

        base_img_name = os.path.basename(img_path).rsplit('.', 1)[0]

        output_filepath_tp = os.path.join(subset_output_dir, f"{base_img_name}_TP.jpg")
        cv2.imwrite(output_filepath_tp, cv2.cvtColor(img_tp, cv2.COLOR_RGB2BGR))
        print(f"    Saved True Positive visualization to {output_filepath_tp}")

        output_filepath_fp = os.path.join(subset_output_dir, f"{base_img_name}_FP.jpg")
        cv2.imwrite(output_filepath_fp, cv2.cvtColor(img_fp, cv2.COLOR_RGB2BGR))
        print(f"    Saved False Positive visualization to {output_filepath_fp}")

        output_filepath_fn = os.path.join(subset_output_dir, f"{base_img_name}_FN.jpg")
        cv2.imwrite(output_filepath_fn, cv2.cvtColor(img_fn, cv2.COLOR_RGB2BGR))
        print(f"    Saved False Negative visualization to {output_filepath_fn}")

print("\nImage visualization process complete!")

## Confusion matrix heatmaps (PCB-MC-M, missing components)

From `Confusion_Matrix_Missing_Components.ipynb`. The GT-vs-prediction
matching and cross-fold accumulation (`build_confusion_matrix`) now live in
`scripts/reproduce_paper_tables.py`; this section imports it to get `cm_all`
and renders the two publication figures.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics --quiet
!pip install seaborn --quiet

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '/content/pcb-mc/scripts')  # adjust to your checkout path
from reproduce_paper_tables import build_confusion_matrix, MISSING_CLASSES

DATA_ROOT = '/content/drive/MyDrive/PCB_MC/Data/'
SUBSET = 'missing_only'
N_FOLDS = 5
MODEL_NAME = 'yolov11'

# Predicted YOLO label dir per fold (already produced by running inference
# with the trained checkpoint, e.g. via ultralytics `model.predict(..., save_txt=True, save_conf=True)`)
PRED_DIR_PATTERN = '/content/cm_preds/{}_fold_{{}}/labels'.format(MODEL_NAME)

IOU_THRESHOLD = 0.5
OUTPUT_DIR = '/content/drive/MyDrive/PCB_MC/figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

from pathlib import Path
pred_label_dirs = {
    fold: Path(PRED_DIR_PATTERN.format(fold))
    for fold in range(N_FOLDS)
    if os.path.exists(PRED_DIR_PATTERN.format(fold))
}

cm_all, cm_per_fold, per_class_df = build_confusion_matrix(
    Path(DATA_ROOT), SUBSET, pred_label_dirs, MISSING_CLASSES, IOU_THRESHOLD
)
display(per_class_df)

In [ ]:
short_names = [c.replace('Missing ', 'M. ') for c in MISSING_CLASSES]
labels = short_names + ['Background']


def plot_confusion_matrix(cm, labels, title, save_path, normalize=False,
                          figsize=(10, 8), cmap='Blues', fontsize=9):
    """
    Plot a confusion matrix as a heatmap.

    Args:
        cm: confusion matrix (numpy array)
        labels: list of class labels
        title: plot title
        save_path: where to save the figure
        normalize: if True, normalize rows to percentages
        figsize: figure size
        cmap: colormap
        fontsize: font size for annotations
    """
    fig, ax = plt.subplots(figsize=figsize)

    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True).astype(float)
        row_sums[row_sums == 0] = 1
        cm_plot = (cm.astype(float) / row_sums) * 100
        fmt = '.1f'
    else:
        cm_plot = cm.astype(float)
        fmt = '.0f'

    annot = np.empty_like(cm_plot, dtype=object)
    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            val = cm_plot[i, j]
            if normalize:
                annot[i, j] = f'{val:.1f}%' if val > 0 else ''
            else:
                annot[i, j] = f'{int(val)}' if val > 0 else ''

    sns.heatmap(
        cm_plot, annot=annot, fmt='', cmap=cmap,
        xticklabels=labels, yticklabels=labels, square=True,
        linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.8},
        annot_kws={'size': fontsize}, ax=ax,
    )

    ax.set_xlabel('Predicted Class', fontsize=12, fontweight='bold')
    ax.set_ylabel('Ground Truth', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


plot_confusion_matrix(
    cm=cm_all, labels=labels,
    title=f'Confusion Matrix — {MODEL_NAME.upper()} on PCB-MC-M (5-fold, counts)',
    save_path=os.path.join(OUTPUT_DIR, f'cm_{MODEL_NAME}_counts.png'),
    normalize=False, cmap='Blues',
)

plot_confusion_matrix(
    cm=cm_all, labels=labels,
    title=f'Confusion Matrix — {MODEL_NAME.upper()} on PCB-MC-M (5-fold, normalized)',
    save_path=os.path.join(OUTPUT_DIR, f'cm_{MODEL_NAME}_normalized.png'),
    normalize=True, cmap='Blues',
)

In [ ]:
def plot_paper_cm(cm_full, class_names, model_name, save_path,
                  include_background=True, figsize=(8, 7)):
    """
    Generate a publication-quality confusion matrix figure.
    """
    n = len(class_names)

    if include_background:
        cm = cm_full.copy()
        display_names = [c.replace('Missing ', '') for c in class_names] + ['BG']
    else:
        cm = cm_full[:n, :n].copy()
        display_names = [c.replace('Missing ', '') for c in class_names]

    row_sums = cm.sum(axis=1, keepdims=True).astype(float)
    row_sums[row_sums == 0] = 1
    cm_norm = (cm / row_sums) * 100

    annot = np.empty_like(cm_norm, dtype=object)
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            val = cm_norm[i, j]
            if val >= 1:
                annot[i, j] = f'{val:.0f}'
            elif val > 0:
                annot[i, j] = f'{val:.1f}'
            else:
                annot[i, j] = ''

    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        cm_norm, annot=annot, fmt='', cmap='Blues',
        xticklabels=display_names, yticklabels=display_names, square=True,
        linewidths=0.8, linecolor='white', vmin=0, vmax=100,
        cbar_kws={'shrink': 0.7, 'label': 'Percentage (%)'},
        annot_kws={'size': 10, 'fontweight': 'bold'}, ax=ax,
    )

    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Ground Truth', fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


plot_paper_cm(
    cm_full=cm_all, class_names=MISSING_CLASSES, model_name=MODEL_NAME,
    save_path=os.path.join(OUTPUT_DIR, f'cm_{MODEL_NAME}_paper_with_bg.pdf'),
    include_background=True, figsize=(8, 7),
)

plot_paper_cm(
    cm_full=cm_all, class_names=MISSING_CLASSES, model_name=MODEL_NAME,
    save_path=os.path.join(OUTPUT_DIR, f'cm_{MODEL_NAME}_paper_no_bg.pdf'),
    include_background=False, figsize=(7, 6),
)